<a href="https://colab.research.google.com/github/Daniel-534/ModelamientoNCuerpos/blob/main/S_Stars_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evolución dinámica e interacciones gravitacionales del cúmulo de estrellas S en el entorno del agujero negro supermasivo Sgr A*

# Modelado N Cuerpos — Mecánica Celeste

## Fuentes

### Artículos científicos

* [An Update on Monitoring Stellar Orbits in the Galactic Center (Gillessen et al. 2017)](https://arxiv.org/abs/1611.09144)
* [Kinematic Structure of the Galactic Center S-cluster (Ali et al. 2020)](https://arxiv.org/abs/2006.11399)
* [Sagittarius A* - The Milky Way Supermassive Black Hole (GRAVITY Collaboration 2022)](https://arxiv.org/abs/2503.20081)
* [Detection of the Schwarzschild Precession in the Orbit of the Star S2 near the Galactic Centre Massive Black Hole (GRAVITY Collaboration 2020)](https://arxiv.org/abs/2004.07187)

### Bases de datos

* [25yrs monitoring of stellar orbits in the GC (Gillessen+, 2017)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/837/30)
* [Kinematic structure of the Galactic Center S cluster (Ali+, 2020)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/896/100)

### Herramientas computacionales

* [`pymcel`](https://pypi.org/project/pymcel/) — Mecánica Celeste en Python (Zuluaga et al.)
* [`scipy.integrate.odeint`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.odeint.html) — Integración de ODEs (Adams-Moulton)
* [`plotly`](https://plotly.com/python/) — Visualizaciones interactivas 3D
* [`matplotlib`](https://matplotlib.org/) — Gráficos y animaciones

## Preguntas de investigación

1. **Dinámica Newtoniana base:** ¿Cómo evolucionan las órbitas de las estrellas S
   (subconjunto de 5, luego las 39 del catálogo) al integrar numéricamente sus estados
   cartesianos iniciales bajo la influencia dominante de Sgr A*?

2. **Perturbaciones de N-cuerpos:** ¿Qué tan significativas son las interacciones
   gravitacionales estrella–estrella frente a la atracción central de Sgr A*?
   Comparación entre *N* simulaciones de 2 cuerpos (estrella + Sgr A*) y una única
   simulación de *N+1* cuerpos.

3. **Estadística orbital:** ¿Cuál es la distribución de los parámetros orbitales
   (semieje mayor, excentricidad, inclinación) de las estrellas S? ¿Difieren
   sistemáticamente los discos negro y rojo?

4. **Ley de Kepler:** ¿Se cumple la tercera ley de Kepler ($T^2 \propto a^3$) en el
   sistema simulado? ¿Hay desviaciones debidas a las perturbaciones mutuas?

5. **Conservación de magnitudes:** ¿Cómo se conserva la energía total y el momento
   angular de cada estrella a lo largo de la simulación numérica?

6. **Velocidades y acercamientos:** ¿Cuáles son las velocidades máximas alcanzadas
   durante los periastros? ¿Qué estrella pasa más cerca de Sgr A*?

In [ ]:
!pip install -Uq pymcel celluloid
!apt-get install -y ffmpeg 2>&1 | grep -E "(ffmpeg|already|Error|error)" || echo "ffmpeg listo"

In [ ]:
import pymcel as pc
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import pandas as pd
from IPython.display import HTML, display, Video
import warnings
warnings.filterwarnings('ignore')

# Estilo global de matplotlib
plt.rcParams.update({
    'figure.facecolor': '#0d0d1a',
    'axes.facecolor':   '#0d0d1a',
    'axes.edgecolor':   '#555577',
    'axes.labelcolor':  '#ccccee',
    'xtick.color':      '#ccccee',
    'ytick.color':      '#ccccee',
    'text.color':       '#ccccee',
    'grid.color':       '#333355',
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#555577',
    'font.size':        11,
})
print("Importaciones completadas.")

---
## 1. Fundamentos Teóricos

### 1.1 Problema de N cuerpos gravitacional

El sistema de estrellas S en el entorno de Sgr A* es un ejemplo canónico del
**problema de N cuerpos gravitacional**. Las ecuaciones de movimiento para el cuerpo $i$
(con $i = 0, 1, \ldots, N-1$) son:

$$\ddot{\mathbf{r}}_i = -G \sum_{j=0,\, j \neq i}^{N-1}
\frac{m_j\,(\mathbf{r}_i - \mathbf{r}_j)}{|\mathbf{r}_i - \mathbf{r}_j|^3}$$

donde $G$ es la constante de gravitación universal, $m_j$ la masa del cuerpo $j$ y
$\mathbf{r}_i$ su vector de posición.

### 1.2 Dominancia de Sgr A*

Dado que $M_{\rm Sgr\,A*} \approx 4.3 \times 10^6\,M_\odot$ y las estrellas S tienen
masas $m_\star \in [8, 14]\,M_\odot$, la aceleración sobre la estrella $i$ se descompone en:

$$\ddot{\mathbf{r}}_i = \underbrace{-\frac{G M_{\rm Sgr\,A*}}{r_i^3}\,\mathbf{r}_i}_{\text{término kepleriano}}
+ \underbrace{\sum_{j \neq 0,\, j \neq i} \frac{G m_j\,(\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}}_{\text{perturbaciones mutuales}}$$

La razón de fuerzas característica es:

$$\frac{F_{\star\star}}{F_{\rm Sgr\,A*}} \sim \frac{N_{\rm est}\,\langle m_\star \rangle}{4\,M_{\rm Sgr\,A*}}
\approx \frac{39 \times 11\,M_\odot}{4 \times 4.3 \times 10^6\,M_\odot} \sim 2.5 \times 10^{-5}$$

### 1.3 Elementos orbitales clásicos

Cada órbita kepleriana queda determinada por los seis **elementos orbitales de Keplero**:

| Símbolo | Nombre | Significado físico |
|---------|--------|-------------------|
| $a$ | Semieje mayor | Tamaño de la elipse |
| $e$ | Excentricidad | Forma ($e=0$ círculo, $e\to 1$ parabólica) |
| $i$ | Inclinación | Ángulo del plano orbital respecto al ecuador |
| $\Omega$ | Longitud del nodo ascendente | Orientación del nodo en el ecuador |
| $\omega$ | Argumento del periastro | Orientación del periastro en el plano orbital |
| $f$ | Anomalía verdadera | Posición actual en la órbita |

La **tercera ley de Kepler** relaciona período y semieje:

$$T = 2\pi \sqrt{\frac{a^3}{G M_{\rm Sgr\,A*}}}$$

### 1.4 Conservación de energía y momento angular

Para cada estrella $i$ en el campo de Sgr A* (2 cuerpos):

$$E_i = \frac{1}{2} m_i v_i^2 - \frac{G M_{\rm Sgr\,A*}\, m_i}{r_i} = -\frac{G M_{\rm Sgr\,A*}\, m_i}{2a_i}$$

$$\mathbf{L}_i = m_i\,(\mathbf{r}_i \times \mathbf{v}_i), \quad |L_i| = m_i \sqrt{G M_{\rm Sgr\,A*}\, a_i (1-e_i^2)}$$

En el sistema de N+1 cuerpos, $E_{\rm tot}$ y $\mathbf{L}_{\rm tot}$ son invariantes del movimiento
(verificables numéricamente como medida de la calidad de la integración).

### 1.5 Integrador numérico: `pymcel.ncuerpos_solucion`

La función `pc.ncuerpos_solucion(sistema, ts)` utiliza internamente
`scipy.integrate.odeint` con el método de Adams-Moulton (predictor-corrector adaptativo,
de orden variable). Devuelve:

* `rs` — posiciones `(N_cuerpos, N_pasos, 3)` en mpc
* `vs` — velocidades `(N_cuerpos, N_pasos, 3)` en mpc/yr
* `const` — diccionario con arrays de energía cinética `K`, potencial `U`, y momento angular `L`

## Condiciones Iniciales

## 2. Condiciones Iniciales

### 2.1 Base de datos observacional

La base de datos empleada es un archivo `.tsv` con los elementos orbitales de **39 estrellas S**
del cúmulo estelar cercano a Sagitario A* (Sgr A*). Estas estrellas, llamadas colectivamente
*cúmulo S*, orbitan el agujero negro supermasivo central de la Vía Láctea.

El catálogo incluye dos subpoblaciones:
- **Disco negro** (*black disk*): estrellas del cúmulo *interno*, con semiejes menores
  ($a \lesssim 50$ mpc). Incluye a S2, la más estudiada.
- **Disco rojo** (*red disk*): estrellas del cúmulo *externo*, con semiejes mayores.

Cada registro contiene: nombre, tipo de disco, semieje mayor $a$ (mpc) con error $e_a$,
excentricidad $e$ con error $e_e$, inclinación $i$ (°) con error $e_i$,
argumento del periastro $\omega$ (°) con error $e_{\omega}$,
longitud del nodo $\Omega$ (°) con error $e_\Omega$, y época de paso por el periastro $T_{\rm clos}$.

In [ ]:
# Base de datos extraída del repositorio en GitHub (rama principal)
url = "https://raw.githubusercontent.com/Daniel-534/ModelamientoNCuerpos/main/Sgrestrellas.tsv"
df  = pd.read_csv(url, sep=';')
df['Name'] = df['Name'].str.strip()
df['Disk'] = df['Disk'].str.strip()

In [18]:
df.head()

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772


### 2.2 Exploración estadística del catálogo

Antes de simular, realizamos un análisis estadístico de los elementos orbitales para comprender
la estructura del cúmulo S y las diferencias entre el disco negro (interno) y el disco rojo (externo).

In [ ]:
# ─── Estadísticas descriptivas del catálogo ──────────────────────────────────
df_clean = df.copy()
df_clean['a']     = pd.to_numeric(df_clean['a'],     errors='coerce')
df_clean['e']     = pd.to_numeric(df_clean['e'],     errors='coerce')
df_clean['i']     = pd.to_numeric(df_clean['i'],     errors='coerce')
df_clean['omega'] = pd.to_numeric(df_clean['omega'], errors='coerce')
df_clean['Omega'] = pd.to_numeric(df_clean['Omega'], errors='coerce')
df_clean = df_clean.dropna(subset=['a','e','i'])

mask_b = df_clean['Disk'] == 'black'
mask_r = df_clean['Disk'] == 'red'

print("─" * 60)
print(f"Total de estrellas: {len(df_clean)}")
print(f"  Disco negro (interno): {mask_b.sum()}")
print(f"  Disco rojo  (externo): {mask_r.sum()}")
print()
for param, unit in [('a','mpc'), ('e',''), ('i','°')]:
    vals = df_clean[param]
    print(f"{param} [{unit}]:  min={vals.min():.3f}  max={vals.max():.3f}  "
          f"media={vals.mean():.3f}  σ={vals.std():.3f}")
print("─" * 60)

# ─── Figura 1: Distribuciones de parámetros orbitales ─────────────────────────
fig_stat, axes = plt.subplots(2, 3, figsize=(15, 9))
fig_stat.suptitle('Distribución de elementos orbitales — Cúmulo S', fontsize=14, y=1.01)

param_info = [
    ('a',     'Semieje mayor  $a$ (mpc)',       30, '#4FC3F7', '#EF5350'),
    ('e',     'Excentricidad  $e$',             20, '#81C784', '#FF8A65'),
    ('i',     'Inclinación  $i$ (°)',           18, '#CE93D8', '#FFCC02'),
    ('omega', r'Arg. periastro  $\omega$ (°)',  18, '#80DEEA', '#FFAB91'),
    ('Omega', r'Nodo ascendente  $\Omega$ (°)', 18, '#B0BEC5', '#F48FB1'),
]

for ax, (col, xlabel, bins, cb, cr) in zip(axes.flat[:5], param_info):
    ax.hist(df_clean.loc[mask_b, col], bins=bins, alpha=0.75, color=cb, label='Disco negro', edgecolor='white', lw=0.4)
    ax.hist(df_clean.loc[mask_r, col], bins=bins, alpha=0.75, color=cr, label='Disco rojo',  edgecolor='white', lw=0.4)
    ax.set_xlabel(xlabel); ax.set_ylabel('Conteo')
    ax.legend(fontsize=8); ax.grid(True)

# Scatter a vs e
ax6 = axes.flat[5]
sc1 = ax6.scatter(df_clean.loc[mask_b,'a'], df_clean.loc[mask_b,'e'],
                  c='#4FC3F7', s=60, alpha=0.85, edgecolors='white', lw=0.4, label='Disco negro')
sc2 = ax6.scatter(df_clean.loc[mask_r,'a'], df_clean.loc[mask_r,'e'],
                  c='#EF5350', s=60, alpha=0.85, edgecolors='white', lw=0.4, label='Disco rojo')
for _, row in df_clean[df_clean['Name'].isin(['S2','S62','S55','S175'])].iterrows():
    ax6.annotate(row['Name'], (row['a'], row['e']), fontsize=7.5,
                 xytext=(4,4), textcoords='offset points', color='#FFEE58')
ax6.set_xlabel('$a$ (mpc)'); ax6.set_ylabel('Excentricidad $e$')
ax6.set_title('Diagrama $a$ vs $e$'); ax6.legend(fontsize=8); ax6.grid(True)

plt.tight_layout()
plt.savefig('orbital_elements_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 1 — Distribución de elementos orbitales guardada.")

### 2.3 Análisis de períodos orbitales y ley de Kepler

La tercera ley de Kepler en el campo de Sgr A* predice:

$$T = 2\pi \sqrt{\frac{a^3}{G M_{\rm Sgr\,A*}}} \quad \Longrightarrow \quad T^2 = \frac{4\pi^2}{G M_{\rm Sgr\,A*}} \cdot a^3$$

Calculamos $T$ para cada estrella y verificamos que $T^2 \propto a^3$.

In [ ]:
# ─── Períodos orbitales (3a ley de Kepler) ────────────────────────────────────
# Primero necesitamos G_canon y mu_SgrA (se definen más adelante si el cuaderno
# se ejecuta en orden; aquí los recalculamos localmente para la celda de análisis)
G_SI_loc  = 6.674e-11
mpc_m_loc = 3.0857e13
Msol_loc  = 1.989e30
yr_s_loc  = 3.1557e7
G_loc     = G_SI_loc * Msol_loc * yr_s_loc**2 / mpc_m_loc**3
mu_loc    = G_loc * 4.3e6   # mpc³/yr²

a_vals  = df_clean['a'].values
T_vals  = 2 * np.pi * np.sqrt(a_vals**3 / mu_loc)   # años

df_orbits = df_clean[['Name','Disk','a','e','i']].copy()
df_orbits['T_yr']   = T_vals
df_orbits['T_kyr']  = T_vals / 1000
df_orbits = df_orbits.sort_values('T_yr')

print("Períodos orbitales de las 39 estrellas S:")
print(df_orbits[['Name','a','e','T_yr']].to_string(index=False))

# ─── Figura 2: Verificación de la 3ª ley de Kepler ───────────────────────────
fig_kep, axes_kep = plt.subplots(1, 2, figsize=(13, 5))
fig_kep.suptitle("Tercera Ley de Kepler — Cúmulo S", fontsize=13)

c_b = '#4FC3F7'; c_r = '#EF5350'
for ax in axes_kep:
    ax.scatter(a_vals[mask_b.values[:len(a_vals)]], T_vals[mask_b.values[:len(a_vals)]],
               color=c_b, s=50, alpha=0.85, edgecolors='white', lw=0.4, label='Disco negro')
    ax.scatter(a_vals[mask_r.values[:len(a_vals)]], T_vals[mask_r.values[:len(a_vals)]],
               color=c_r, s=50, alpha=0.85, edgecolors='white', lw=0.4, label='Disco rojo')

# Curva teórica
a_fit  = np.linspace(a_vals.min()*0.8, a_vals.max()*1.05, 200)
T_fit  = 2 * np.pi * np.sqrt(a_fit**3 / mu_loc)
axes_kep[0].plot(a_fit, T_fit, 'w--', lw=1.5, alpha=0.6, label='Kepler teórico')
axes_kep[0].set(xlabel='$a$ (mpc)', ylabel='$T$ (años)', title='$T$ vs $a$')
axes_kep[0].legend(fontsize=8); axes_kep[0].grid(True)

# Escala log-log: T² ∝ a³ → pendiente 1.5 en log(T) vs log(a)
axes_kep[1].scatter(np.log10(a_vals), np.log10(T_vals), s=50, alpha=0.85,
                    c=[c_b if d=='black' else c_r for d in df_clean['Disk']], edgecolors='white', lw=0.4)
a_log  = np.linspace(np.log10(a_vals.min())*0.95, np.log10(a_vals.max())*1.02, 200)
axes_kep[1].plot(a_log, 1.5*a_log + np.log10(2*np.pi/np.sqrt(mu_loc)),
                 'w--', lw=1.5, alpha=0.6, label='Pendiente 3/2')
axes_kep[1].set(xlabel=r'$\log_{10}(a/\mathrm{mpc})$', ylabel=r'$\log_{10}(T/\mathrm{yr})$',
                title=r'Log–log: $T^2 \propto a^3$')
axes_kep[1].legend(fontsize=8); axes_kep[1].grid(True)

# Etiquetas S2, S62
for name in ['S2','S62','S175']:
    row = df_orbits[df_orbits['Name']==name]
    if not row.empty:
        av, tv = float(row['a']), float(row['T_yr'])
        axes_kep[0].annotate(name, (av,tv), xytext=(5,5), textcoords='offset points',
                              fontsize=8, color='#FFEE58')

plt.tight_layout()
plt.savefig('kepler_law_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 2 — Verificación de la ley de Kepler guardada.")

## Conversión de elementos orbitales clásicos a estado cartesiano

Dado un conjunto $(a, e, i, \Omega, \omega)$ medidos observacionalmente, calculamos el semilatus
rectum $p = a(1-e^2)$ y usamos `pymcel.elementos_a_estado(\mu, [p, e, i, \Omega, \omega, f=0])`
para obtener el vector de estado $\mathbf{r}_0, \mathbf{v}_0$ en el periastro ($f=0$).

In [19]:
# Unidades canónicas para el sistema del centro galáctico
# UL = 1 mpc (miliparsec)  |  UT = 1 año  |  UM = 1 Msol
# G se expresa en mpc³/(Msol·yr²)

G_SI   = 6.674e-11        # m³ kg⁻¹ s⁻²
mpc_m  = 3.0857e13        # metros por miliparsec
Msol   = 1.989e30         # kg por masa solar
yr_s   = 3.1557e7         # segundos por año

G_canon = G_SI * Msol * yr_s**2 / mpc_m**3   # mpc³/(Msol·yr²)
print(f"G = {G_canon:.4e} mpc³/(Msol·yr²)")

# Parámetro gravitacional de Sgr A* (μ = G·M)
M_SgrA  = 4.3e6                   # masas solares
mu_SgrA = G_canon * M_SgrA        # mpc³/yr²
print(f"μ(Sgr A*) = {mu_SgrA:.4f} mpc³/yr²")

# pc.elementos_a_estado(mu, [p, e, i, Ω, ω, f]) → [x, y, z, vx, vy, vz]
# p en mpc → posiciones en mpc, velocidades en mpc/yr

In [ ]:
# ─── Conversión orbital: todos los objetos del catálogo ──────────────────────
# Los ángulos (i, Ω, ω) están en grados; se convierten a radianes.
# Condición inicial: periastro (f = 0).  Se excluyen órbitas no ligadas (e ≥ 1).

estados_todos = []   # [x, y, z, vx, vy, vz]  (mpc, mpc/yr)
nombres_todos = []   # etiqueta de cada estrella
disco_todos   = []   # 'black' (cúmulo interno) o 'red' (cúmulo externo)

for _, est in df.iterrows():
    e = float(str(est['e']).strip())
    if e >= 1.0:          # descartar hiperbólicas / parabólicas
        print(f"  Saltando {est['Name'].strip()}: e = {e:.3f}")
        continue

    a = float(str(est['a']).strip())        # semieje mayor (mpc)
    p = a * (1 - e**2)                      # semilatus rectum (mpc)
    i = np.radians(float(str(est['i']).strip()))
    W = np.radians(float(str(est['Omega']).strip()) % 360)   # Ω mod 360°
    w = np.radians(float(str(est['omega']).strip()) % 360)   # ω mod 360°
    f = 0.0                                                   # periastro

    estado = pc.elementos_a_estado(mu_SgrA, [p, e, i, W, w, f])
    estados_todos.append(estado)
    nombres_todos.append(est['Name'].strip())
    disco_todos.append(est['Disk'].strip())

N_estrellas = len(estados_todos)
print(f"\nEstrellas con órbitas ligadas (e < 1): {N_estrellas}")
print(f"  Disco negro (cúmulo interno): {disco_todos.count('black')}")
print(f"  Disco rojo  (cúmulo externo): {disco_todos.count('red')  }")

In [ ]:
columnas_estado = ['x', 'y', 'z', 'vx', 'vy', 'vz']
df_estados = pd.DataFrame(estados_todos, index=nombres_todos, columns=columnas_estado)
df_estados.round(5)

### Masas
El agujero negro supermasivo Sgr A* en el centro galáctico tiene una masa bien determinada de ~4.15×10^6 M☉ con precisión de ~0.3%
, medida principalmente mediante el seguimiento de órbitas estelares cercanas (Genzel, Ghez, GRAVITY). Por su parte, las estrellas S más luminosas del cúmulo interno son jóvenes de tipo espectral ~B0–B3V. Dado que las masas individuales de las estrellas S se estiman mediante su tipo espectral, y su efecto perturbador es mínimo frente a Sgr A, se asignaron masas estimadas de entre 8 y 15 masas solares basadas en la literatura (Gillessen et al., 2017).

In [ ]:
# ─── Masas estelares ──────────────────────────────────────────────────────────
# Las estrellas S son de tipo B0–B3V.  Se asignan masas en [8, 14] Msol con
# semilla fija para garantizar reproducibilidad.
np.random.seed(42)
masas_todos = np.random.uniform(8.0, 14.0, N_estrellas)   # Msol

print(f"Masas asignadas a las {N_estrellas} estrellas:")
for nombre, masa in zip(nombres_todos, masas_todos):
    print(f"  {nombre:6s}: {masa:.2f} Msol")

---
## Respuesta 1 — Dinámica Newtoniana base: subconjunto de 5 estrellas

Validamos el modelo con las 5 primeras estrellas del disco negro (S1, S2, S8, S9, S12).

**Integrador:** `pymcel.ncuerpos_solucion` → `scipy.integrate.odeint` (Adams-Moulton adaptativo).
El array de resultados tiene forma `(N_cuerpos, N_pasos, 3)`.

**Verificación:** la energía total $E = K + U$ debe conservarse (error relativo < 10⁻⁵).

In [ ]:
# ─── Sistema: Sgr A* (origen) + 5 estrellas del disco interno ────────────────
# rs tiene forma (N_cuerpos, N_pasos, 3): rs[i, t, xyz]
n_demo = 5
est_5  = estados_todos[:n_demo]
nom_5  = nombres_todos[:n_demo]
mas_5  = masas_todos[:n_demo]

sistema_5 = [dict(m=mu_SgrA, r=[0., 0., 0.], v=[0., 0., 0.])]   # Sgr A*
for estado, masa in zip(est_5, mas_5):
    x, y, z, vx, vy, vz = estado
    sistema_5.append(dict(m=G_canon * masa, r=[x, y, z], v=[vx, vy, vz]))

print(f"Sistema con {len(sistema_5)} cuerpos:")
print(f"  {'Sgr A*':8s}: μ = {mu_SgrA:.4f} mpc³/yr²")
for nombre, s, masa in zip(nom_5, sistema_5[1:], mas_5):
    r0, v0 = np.linalg.norm(s['r']), np.linalg.norm(s['v'])
    print(f"  {nombre:8s}: μ = {G_canon*masa:.2e} mpc³/yr², "
          f"|r₀| = {r0:.3f} mpc, |v₀| = {v0:.3f} mpc/yr")

# ─── Integración numérica ─────────────────────────────────────────────────────
T_sim_5   = 100
N_pasos_5 = 5000
ts_5      = np.linspace(0, T_sim_5, N_pasos_5)

print(f"\nIntegrando {T_sim_5} años con {N_pasos_5} pasos (dt = {T_sim_5/N_pasos_5:.3f} yr) ...")
rs_5, vs_5, _, _, const_5 = pc.ncuerpos_solucion(sistema_5, ts_5)
# rs_5.shape = (N_cuerpos, N_pasos, 3)
print(f"¡Integración completada!  rs_5.shape = {rs_5.shape}")

# ─── Conservación de energía ──────────────────────────────────────────────────
K_5, U_5 = const_5['K'], const_5['U']
E_5      = K_5 + U_5
err_E_5  = np.abs((E_5 - E_5[0]) / E_5[0]).max()
print(f"Error relativo máx. en energía: {err_E_5:.2e}")

fig_E5, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ts_5, (E_5 - E_5[0]) / np.abs(E_5[0]), color='tab:red')
axs[0].set(xlabel="Tiempo (años)", ylabel="(E − E₀) / |E₀|",
           title="Conservación de energía — 5 estrellas S")
axs[0].grid(True)

axs[1].plot(ts_5, K_5, label='K (cinética)',  color='tab:orange')
axs[1].plot(ts_5, U_5, label='U (potencial)', color='tab:blue')
axs[1].plot(ts_5, E_5, label='E = K + U',     color='tab:green', lw=2)
axs[1].set(xlabel="Tiempo (años)", ylabel="Energía (mpc² yr⁻²)",
           title="Componentes de energía")
axs[1].legend(); axs[1].grid(True)
plt.tight_layout(); plt.show()

# ─── Visualización 3D con Plotly ─────────────────────────────────────────────
# rs_5[i, :, :] = trayectoria del cuerpo i; rs_5[i, -1, :] = posición final
COLORS_5 = ['gold', '#4C9BE8', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
nombres_5_full = ['Sgr A*'] + nom_5

fig_5 = go.Figure()
for i, label in enumerate(nombres_5_full):
    x_traj, y_traj, z_traj = rs_5[i, :, 0], rs_5[i, :, 1], rs_5[i, :, 2]
    fig_5.add_trace(go.Scatter3d(
        x=x_traj, y=y_traj, z=z_traj,
        mode='lines',
        line=dict(width=3 if i == 0 else 1.5, color=COLORS_5[i]),
        name=label,
    ))
    # posición final + etiqueta
    fig_5.add_trace(go.Scatter3d(
        x=[x_traj[-1]], y=[y_traj[-1]], z=[z_traj[-1]],
        mode='markers+text',
        marker=dict(size=6 if i > 0 else 10, color=COLORS_5[i],
                    symbol='circle', line=dict(width=1, color='white')),
        text=[label], textfont=dict(size=11, color=COLORS_5[i]),
        textposition='top center', showlegend=False,
    ))

# Anotación para S2 (estrella de referencia más importante)
fig_5.add_annotation(
    text="★ S2 · T≈16 yr · e=0.887",
    showarrow=False, xref="paper", yref="paper",
    x=0.01, y=0.97, font=dict(size=12, color='#E74C3C'),
    bgcolor='rgba(0,0,0,0.45)', borderpad=4,
)

fig_5.update_layout(
    title=f"Sgr A* + 5 estrellas S — simulación de {T_sim_5} años",
    scene=dict(xaxis_title="x (mpc)", yaxis_title="y (mpc)", zaxis_title="z (mpc)",
               aspectmode='data'),
    legend=dict(font=dict(size=11)),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_5.show()

### Visualización 3D de las 5 órbitas — matplotlib

Se representan las trayectorias de las 5 primeras estrellas del disco negro usando
`matplotlib` con fondo oscuro, colores degradados y marcadores en la posición inicial.

In [ ]:
# ─── Figura 3: Órbitas 3D de las 5 estrellas con matplotlib ──────────────────
colors_5  = ['#00E5FF','#69F0AE','#FFEB3B','#FF4081','#B39DDB']
fig3d_5   = plt.figure(figsize=(10, 8))
ax3d_5    = fig3d_5.add_subplot(111, projection='3d')
ax3d_5.set_facecolor('#0d0d1a')
fig3d_5.patch.set_facecolor('#0d0d1a')

# Sgr A* en el origen
ax3d_5.scatter([0],[0],[0], color='gold', s=250, marker='*', zorder=10,
               depthshade=False, label='Sgr A*')

for i, (nombre, color) in enumerate(zip(nom_5, colors_5)):
    traj = rs_5[i+1, :, :]
    ax3d_5.plot(traj[:,0], traj[:,1], traj[:,2],
                color=color, lw=1.2, alpha=0.85, label=nombre)
    ax3d_5.scatter(*traj[0], color=color, s=50, zorder=5, depthshade=False)

ax3d_5.set_xlabel('$x$ (mpc)', labelpad=6)
ax3d_5.set_ylabel('$y$ (mpc)', labelpad=6)
ax3d_5.set_zlabel('$z$ (mpc)', labelpad=6)
ax3d_5.set_title('Trayectorias 3D — 5 estrellas S + Sgr A* (100 años)', pad=12)
ax3d_5.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('orbits_3d_5stars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 3 — Órbitas 3D de las 5 estrellas guardada.")

### Conservación del momento angular (5 estrellas)

Para un sistema de 2 cuerpos (estrella + Sgr A* fijo), el momento angular específico
$\mathbf{h}_i = \mathbf{r}_i \times \mathbf{v}_i$ es una integral de movimiento.
En el sistema de N+1 cuerpos, la perturbación debida a las demás estrellas hace que $\mathbf{h}_i$
evolucione lentamente. Analizamos $|\mathbf{h}_i(t)|$ para las 5 estrellas.

In [ ]:
# ─── Momento angular específico |h(t)| para las 5 estrellas ──────────────────
# h_i(t) = r_i(t) × v_i(t),  cuerpo i+1 en rs_5/vs_5
fig_Lz, axes_L = plt.subplots(1, 2, figsize=(13, 5))
fig_Lz.suptitle("Conservación del momento angular — 5 estrellas S", fontsize=12)

for i, (nombre, color) in enumerate(zip(nom_5, colors_5)):
    r_i = rs_5[i+1, :, :]
    v_i = vs_5[i+1, :, :]
    h_i = np.cross(r_i, v_i)       # (N_pasos, 3)
    h_mag = np.linalg.norm(h_i, axis=1)
    h0    = h_mag[0]
    axes_L[0].plot(ts_5, h_mag, color=color, label=nombre, lw=1.3)
    axes_L[1].plot(ts_5, (h_mag - h0)/h0, color=color, label=nombre, lw=1.3)

axes_L[0].set(xlabel='Tiempo (años)', ylabel=r'$|\mathbf{h}|$ (mpc² yr⁻¹)',
              title='Módulo del momento angular específico')
axes_L[1].set(xlabel='Tiempo (años)', ylabel=r'$(|h| - h_0)/h_0$',
              title=r'Variación relativa de $|\mathbf{h}|$')
for ax in axes_L: ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.savefig('angular_momentum_5stars.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Dirección del vector h: evolución del polo orbital ─────────────────────
fig_pole, ax_pole = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
ax_pole.set_facecolor('#0d0d1a')
fig_pole.patch.set_facecolor('#0d0d1a')
for i, (nombre, color) in enumerate(zip(nom_5, colors_5)):
    r_i = rs_5[i+1, :, :]
    v_i = vs_5[i+1, :, :]
    h_i = np.cross(r_i, v_i)
    h_i /= (np.linalg.norm(h_i, axis=1, keepdims=True) + 1e-30)
    theta = np.arctan2(h_i[:,1], h_i[:,0])
    rho   = np.arccos(np.clip(h_i[:,2], -1, 1))
    ax_pole.plot(theta, np.degrees(rho), color=color, alpha=0.7, lw=1, label=nombre)
    ax_pole.scatter(theta[0], np.degrees(rho[0]), color=color, s=30, zorder=5)
ax_pole.set_title(r"Polo orbital $\hat{h}$ en coordenadas esféricas (t = 0–100 yr)", pad=15)
ax_pole.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig('orbital_pole_5stars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 4–5 — Momento angular guardadas.")

---
## Respuesta 2 — Perturbaciones N-cuerpos vs. 2 cuerpos (subconjunto de 5 estrellas)

Para cuantificar el efecto de las interacciones estrella–estrella, comparamos dos modelos:

| Modelo | Descripción |
|--------|-------------|
| **2 cuerpos** | Cada estrella se integra **por separado** con Sgr A* fijo en el origen (órbita kepleriana pura) |
| **N+1 cuerpos** | Todas las estrellas se integran **simultáneamente**; cada par interactúa gravitacionalmente |

La desviación $\Delta r(t) = |\mathbf{r}_{N+1}(t) - \mathbf{r}_\text{Kepler}(t)|$ mide
la magnitud acumulada de las perturbaciones estelares.

In [ ]:
# ─── Trayectorias kepleranas puras (2 cuerpos) para cada una de las 5 estrellas ─
# rs_2b tiene forma (2, N_pasos, 3): cuerpo 0 = Sgr A*, cuerpo 1 = estrella
print("Calculando trayectorias de 2 cuerpos...")
rs_2b_5 = []   # lista de arrays (N_pasos, 3), una por estrella

for i, (nombre, estado, masa) in enumerate(zip(nom_5, est_5, mas_5)):
    sistema_2b = [
        dict(m=mu_SgrA,        r=[0., 0., 0.],     v=[0., 0., 0.]),
        dict(m=G_canon * masa,  r=list(estado[:3]), v=list(estado[3:])),
    ]
    rs_2b_i, _, _, _, _ = pc.ncuerpos_solucion(sistema_2b, ts_5)
    rs_2b_5.append(rs_2b_i[1, :, :])   # cuerpo 1 = estrella
    print(f"  {nombre}: ✓")

# ─── Desviación relativa Δr(t) / |r(t)| ──────────────────────────────────────
fig_comp5, axes = plt.subplots(1, 2, figsize=(14, 5))

delta_max_5 = []
for i, nombre in enumerate(nom_5):
    r_nb   = rs_5[i + 1, :, :]                               # N+1 cuerpos: cuerpo i+1
    r_2b   = rs_2b_5[i]                                      # 2 cuerpos (Kepler)
    dr     = np.linalg.norm(r_nb - r_2b, axis=1)             # |Δr| (mpc)
    dr_rel = dr / (np.linalg.norm(r_nb, axis=1) + 1e-30)     # relativo

    axes[0].plot(ts_5, dr,     label=nombre)
    axes[1].plot(ts_5, dr_rel, label=nombre)
    delta_max_5.append(dr_rel.max())

for ax, ylabel, title in zip(
        axes,
        ["Δr (mpc)", "Δr / |r|  (adimensional)"],
        ["Desviación absoluta: N-cuerpos vs. 2-cuerpos",
         "Desviación relativa: N-cuerpos vs. 2-cuerpos"]):
    ax.set(xlabel="Tiempo (años)", ylabel=ylabel, title=title)
    ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

# ─── Estimación analítica de la relación de fuerzas ──────────────────────────
# F_est / F_SgrA ≈ (N × <m_est>) / (4 × M_SgrA)  (distancias comparables)
ratio_5 = n_demo * np.mean(mas_5) / (4 * 4.3e6)
print(f"\nEstimación analítica de relación de fuerzas: {ratio_5:.2e}")
print(f"→ Las perturbaciones representan ~ {ratio_5*100:.4f}% de la fuerza de Sgr A*")
print(f"Desviación relativa máxima media: {np.mean(delta_max_5):.2e}")

### Interpretación física de las perturbaciones (5 estrellas)

Las curvas de desviación relativa $\Delta r / |r|$ revelan dos regímenes:

1. **Oscilaciones rápidas de corto período:** causadas por los acercamientos entre
   estrellas en el periastro. La amplitud es proporcional a $m_j / (d_{ij}^2 M_{\rm Sgr A*})$.

2. **Deriva secular a largo plazo:** acumulación coherente de impulsos que distorsiona
   la elipse kepleriana, análoga a la precesión del perihelio.

La estimación analítica $F_{\star\star}/F_{\rm Sgr A*} \sim 10^{-5}$ predice que las
perturbaciones son **despreciables en escalas de décadas** pero se vuelven
significativas en $\sim 10^4$ años de integración.

---
## Escalado completo — 39 estrellas S + Sgr A*

Con las condiciones iniciales de las 39 estrellas del catálogo, integramos el sistema completo de
40 cuerpos durante 100 años.

> **Nota sobre el paso temporal:** la estrella de período más corto en el cúmulo interno es S2
> (T ≈ 16 yr) y S62 tiene e = 0.98.  Con 5 000 pasos en 100 años (dt ≈ 7 días) se logra una
> precisión energética típicamente mejor que 10⁻⁵.

In [ ]:
# ─── Sistema completo: Sgr A* + N_estrellas estrellas S ──────────────────────
sistema_39 = [dict(m=mu_SgrA, r=[0., 0., 0.], v=[0., 0., 0.])]   # Sgr A*
for estado, masa in zip(estados_todos, masas_todos):
    x, y, z, vx, vy, vz = estado
    sistema_39.append(dict(m=G_canon * masa, r=[x, y, z], v=[vx, vy, vz]))

n_cuerpos_39 = len(sistema_39)
T_sim_39     = 100
N_pasos_39   = 5000
ts_39        = np.linspace(0, T_sim_39, N_pasos_39)

print(f"Sistema: {n_cuerpos_39} cuerpos  (Sgr A* + {N_estrellas} estrellas S)")
print(f"Integrando {T_sim_39} años con {N_pasos_39} pasos (dt = {T_sim_39/N_pasos_39:.3f} yr) ...")
rs_39, vs_39, _, _, const_39 = pc.ncuerpos_solucion(sistema_39, ts_39)
# rs_39.shape = (n_cuerpos_39, N_pasos_39, 3)
print(f"¡Integración completada!  rs_39.shape = {rs_39.shape}")

# ─── Conservación de energía ──────────────────────────────────────────────────
K_39, U_39 = const_39['K'], const_39['U']
E_39       = K_39 + U_39
err_E_39   = np.abs((E_39 - E_39[0]) / E_39[0]).max()
print(f"Error relativo máx. en energía ({N_estrellas} estrellas): {err_E_39:.2e}")

fig_E39, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ts_39, (E_39 - E_39[0]) / np.abs(E_39[0]), color='tab:red')
axs[0].set(xlabel="Tiempo (años)", ylabel="(E − E₀) / |E₀|",
           title=f"Conservación de energía — {N_estrellas} estrellas S")
axs[0].grid(True)

axs[1].plot(ts_39, K_39, label='K', color='tab:orange')
axs[1].plot(ts_39, U_39, label='U', color='tab:blue')
axs[1].plot(ts_39, E_39, label='E=K+U', color='tab:green', lw=2)
axs[1].set(xlabel="Tiempo (años)", ylabel="Energía (mpc² yr⁻²)",
           title="Componentes de energía")
axs[1].legend(); axs[1].grid(True)
plt.tight_layout(); plt.show()

---
## 3. Análisis de Velocidades y Distancias al Centro Galáctico

La ecuación de la vis-viva relaciona la velocidad orbital con la posición:

$$v^2 = G M_{\rm Sgr A*} \left(\frac{2}{r} - \frac{1}{a}\right)$$

La velocidad máxima se alcanza en el **periastro** ($r_{\rm min} = a(1-e)$):

$$v_{\rm max} = \sqrt{G M_{\rm Sgr A*} \frac{1+e}{a(1-e)}}$$

Para S2: $v_{\rm max} \approx \sqrt{G M_{\rm Sgr A*} \frac{1.887}{5.034 \times 0.113}} \approx 7.7$ mpc/yr $\approx 7{,}500$ km/s

Analizamos la velocidad y distancia en función del tiempo para las estrellas más destacadas.

In [ ]:
# ─── Figura 6: Velocidades y distancias — estrellas destacadas ────────────────
# Identificar índices de estrellas destacadas
stars_dest = ['S2', 'S62', 'S55', 'S1', 'S8']
idxs_dest  = []
for sn in stars_dest:
    if sn in nombres_todos:
        idxs_dest.append((sn, nombres_todos.index(sn)+1))  # +1 por Sgr A*

fig_vel, axes_vel = plt.subplots(2, 2, figsize=(14, 10))
fig_vel.suptitle("Velocidades y distancias al centro galáctico — Estrellas destacadas", fontsize=12)
ax_v, ax_r, ax_vr, ax_ee = axes_vel.flat

palette_dest = ['#00E5FF','#FF4081','#FFEB3B','#69F0AE','#B39DDB']

for (sn, si), col in zip(idxs_dest, palette_dest):
    r_i  = rs_39[si, :, :]
    v_i  = vs_39[si, :, :]
    r_mag = np.linalg.norm(r_i, axis=1)
    v_mag = np.linalg.norm(v_i, axis=1)
    ax_v.plot(ts_39, v_mag, color=col, lw=1.2, label=sn, alpha=0.9)
    ax_r.plot(ts_39, r_mag, color=col, lw=1.2, label=sn, alpha=0.9)
    ax_vr.scatter(r_mag[::20], v_mag[::20], color=col, s=8, alpha=0.6, label=sn)

ax_v.set(xlabel='Tiempo (años)', ylabel='$|v|$ (mpc/yr)', title='Velocidad orbital vs tiempo')
ax_r.set(xlabel='Tiempo (años)', ylabel='$r$ (mpc)', title='Distancia a Sgr A* vs tiempo')
ax_vr.set(xlabel='$r$ (mpc)', ylabel='$|v|$ (mpc/yr)', title='Diagrama vis-viva $v(r)$')
for ax in [ax_v, ax_r, ax_vr]: ax.legend(fontsize=8); ax.grid(True)

# Curva vis-viva teórica para S2
if 'S2' in nombres_todos:
    i_S2 = nombres_todos.index('S2')
    a_S2 = float(df_clean[df_clean['Name']=='S2']['a'].values[0])
    r_fit = np.linspace(a_S2*0.113*0.95, a_S2*1.887*1.02, 200)
    v_vv  = np.sqrt(np.maximum(mu_SgrA*(2/r_fit - 1/a_S2), 0))
    ax_vr.plot(r_fit, v_vv, 'w--', lw=1.5, alpha=0.7, label='vis-viva S2')
    ax_vr.legend(fontsize=8)

# ─── Velocidades máximas (periastro) para todas las estrellas ────────────────
v_max_all = []
for i in range(len(nombres_todos)):
    v_i   = vs_39[i+1, :, :]
    v_max_all.append(np.linalg.norm(v_i, axis=1).max())

bar_cols = ['#4FC3F7' if d=='black' else '#EF5350' for d in disco_todos]
ax_ee.barh(nombres_todos, v_max_all, color=bar_cols, edgecolor='white', lw=0.4)
ax_ee.set(xlabel='$v_{\rm máx}$ (mpc/yr)', title='Velocidad máxima por estrella')
ax_ee.invert_yaxis()
ax_ee.grid(True, axis='x', alpha=0.5)
patch_b = mpatches.Patch(color='#4FC3F7', label='Disco negro')
patch_r = mpatches.Patch(color='#EF5350', label='Disco rojo')
ax_ee.legend(handles=[patch_b, patch_r], fontsize=8)

plt.tight_layout()
plt.savefig('velocities_distances.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 6 — Análisis de velocidades y distancias guardada.")

---
## 4. Proyecciones Bidimensionales de las Órbitas

Las proyecciones del sistema 3D sobre los planos coordenados (XY, XZ, YZ) permiten
identificar la estructura de disco del cúmulo S y la orientación orbital de cada estrella.
- **Plano XY** (plano del cielo): lo que observamos proyectado sobre el plano de la galaxia.
- **Plano XZ** y **YZ**: perfil orbital, mostrando la extensión fuera del plano.

In [ ]:
# ─── Figura 7: Proyecciones 2D de las 39 órbitas ────────────────────────────
fig_proj, axes_proj = plt.subplots(1, 3, figsize=(18, 6))
fig_proj.suptitle("Proyecciones 2D de las órbitas — 39 estrellas S + Sgr A*", fontsize=13)

proj_labels = [('x','y','Plano XY  (plano del cielo)'),
               ('x','z','Plano XZ  (perfil E–W)'),
               ('y','z','Plano YZ  (perfil N–S)')]

def _col39(idx):
    if idx == 0: return 'gold'
    d = disco_todos[idx - 1]
    return '#4FC3F7' if d == 'black' else '#EF5350'

for ax, (xi, yi, title) in zip(axes_proj, proj_labels):
    coord = {'x':0, 'y':1, 'z':2}
    ci, cj = coord[xi], coord[yi]
    for i in range(n_cuerpos_39):
        col = _col39(i)
        lw  = 2.5 if i == 0 else 0.7
        al  = 1.0 if i == 0 else 0.55
        ax.plot(rs_39[i, :, ci], rs_39[i, :, cj], color=col, lw=lw, alpha=al)
    # Sgr A* en el origen
    ax.scatter([0],[0], color='gold', s=200, marker='*', zorder=10)
    ax.set(xlabel=f'${xi}$ (mpc)', ylabel=f'${yi}$ (mpc)', title=title)
    ax.set_aspect('equal'); ax.grid(True)

# Leyenda global
patch_b = mpatches.Patch(color='#4FC3F7', label='Disco negro')
patch_r = mpatches.Patch(color='#EF5350', label='Disco rojo')
patch_g = mpatches.Patch(color='gold',    label='Sgr A*')
fig_proj.legend(handles=[patch_g, patch_b, patch_r], loc='lower center',
                ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
plt.savefig('orbits_2d_projections.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 7 — Proyecciones 2D guardadas.")

---
## 5. Visualización Interactiva 3D con Plotly

La figura siguiente muestra una visualización tridimensional interactiva del sistema
completo (Sgr A* + 39 estrellas S) durante 100 años de simulación:

- **Trayectorias** (líneas estáticas): órbitas completas integradas numéricamente.
- **Marcadores** (animados): posición instantánea de cada cuerpo en el tiempo $t$.
- **Slider temporal** y botones **▶ Reproducir / ⏸ Pausar**.
- Disco negro en tonos azul/verde; disco rojo en tonos rojo/naranja; Sgr A* en dorado.
- La animación incluye 80 fotogramas que cubren los 100 años de simulación.

In [ ]:
# ─── Paleta de colores por disco ─────────────────────────────────────────────
_pal_black = ['#4C9BE8','#5DADE2','#1A5276','#2E86C1','#AED6F1',
              '#1B4F72','#7FB3D3','#154360','#52BE80','#117A65',
              '#1E8449','#239B56','#27AE60','#2ECC71','#82E0AA',
              '#A9DFBF','#145A32','#196F3D','#1D8348']
_pal_red   = ['#E74C3C','#C0392B','#922B21','#CB4335','#F1948A',
              '#F5CBA7','#E59866','#CA6F1E','#D4AC0D','#B7950B',
              '#9A7D0A','#7D6608','#F0B27A','#E67E22','#DC7633',
              '#A04000','#7E5109','#B9770E','#FAD7A0','#F8C471']
_cnt_b, _cnt_r = 0, 0

def _color(idx):
    global _cnt_b, _cnt_r
    if idx == 0:
        return 'gold'
    disco = disco_todos[idx - 1]
    if disco == 'black':
        c = _pal_black[_cnt_b % len(_pal_black)]; _cnt_b += 1
    else:
        c = _pal_red[_cnt_r % len(_pal_red)];     _cnt_r += 1
    return c

nombres_39_full = ['Sgr A*'] + nombres_todos
colors_39       = [_color(i) for i in range(n_cuerpos_39)]

# ─── Trazas estáticas (trayectorias completas) ───────────────────────────────
# rs_39[i, :, :] = trayectoria del cuerpo i a lo largo del tiempo
static_traces = []
for i in range(n_cuerpos_39):
    label = nombres_39_full[i]
    static_traces.append(go.Scatter3d(
        x=rs_39[i, :, 0], y=rs_39[i, :, 1], z=rs_39[i, :, 2],
        mode='lines',
        line=dict(width=3 if i == 0 else 1, color=colors_39[i]),
        name=label, opacity=1.0 if i == 0 else 0.55,
    ))

# ─── Trazas animadas (marcador de posición instantánea) ───────────────────────
# rs_39[i, 0, :] = posición inicial del cuerpo i
marker_traces = []
for i in range(n_cuerpos_39):
    label = nombres_39_full[i]
    x0, y0, z0 = rs_39[i, 0, 0], rs_39[i, 0, 1], rs_39[i, 0, 2]
    marker_traces.append(go.Scatter3d(
        x=[x0], y=[y0], z=[z0],
        mode='markers',
        marker=dict(size=9 if i == 0 else 5, color=colors_39[i],
                    symbol='diamond' if i == 0 else 'circle',
                    line=dict(width=1, color='white')),
        name=label, showlegend=False,
        hovertemplate=(f'<b>{label}</b><br>'
                       'x=%{x:.3f} mpc<br>y=%{y:.3f} mpc<br>z=%{z:.3f} mpc<extra></extra>'),
    ))

# ─── Frames de animación (80 fotogramas) ─────────────────────────────────────
# En cada frame solo se actualizan los marcadores (traces n_cuerpos_39 .. 2*n_cuerpos_39-1)
N_frames   = 80
frame_idxs = np.linspace(0, N_pasos_39 - 1, N_frames, dtype=int)

frames = []
for fi, idx in enumerate(frame_idxs):
    # rs_39[i, idx, :] = posición del cuerpo i en el instante ts_39[idx]
    frame_data = [
        go.Scatter3d(x=[rs_39[i, idx, 0]],
                     y=[rs_39[i, idx, 1]],
                     z=[rs_39[i, idx, 2]])
        for i in range(n_cuerpos_39)
    ]
    frames.append(go.Frame(
        data=frame_data,
        traces=list(range(n_cuerpos_39, 2 * n_cuerpos_39)),
        name=str(fi),
    ))

# ─── Slider de tiempo ────────────────────────────────────────────────────────
slider_steps = [
    dict(args=[[str(fi)], dict(frame=dict(duration=80, redraw=True), mode='immediate')],
         label=f'{ts_39[idx]:.0f} yr', method='animate')
    for fi, idx in enumerate(frame_idxs)
]

# ─── Anotaciones de texto para estrellas destacadas ──────────────────────────
star_labels = []
for star, text in [('S2',   '★ S2'),
                   ('S62',  'S62 e=0.98'),
                   ('S55',  'S55'),
                   ('S175', 'S175 e≈0.999')]:
    if star in nombres_39_full:
        si = nombres_39_full.index(star)
        x0, y0, z0 = rs_39[si, 0, 0], rs_39[si, 0, 1], rs_39[si, 0, 2]
        star_labels.append(go.Scatter3d(
            x=[x0], y=[y0], z=[z0],
            mode='text',
            text=[text],
            textfont=dict(size=11, color='white'),
            showlegend=False, name=f'{star}_label',
        ))

# ─── Figura final ─────────────────────────────────────────────────────────────
fig_39 = go.Figure(
    data=static_traces + marker_traces + star_labels,
    frames=frames,
    layout=go.Layout(
        title=dict(text=f'Sgr A* + {N_estrellas} estrellas S — {T_sim_39} años',
                   font=dict(size=15)),
        scene=dict(xaxis_title='x (mpc)', yaxis_title='y (mpc)', zaxis_title='z (mpc)',
                   aspectmode='data'),
        legend=dict(font=dict(size=9), itemsizing='constant'),
        margin=dict(l=0, r=0, t=50, b=130),
        updatemenus=[dict(
            type='buttons', showactive=False, direction='left',
            x=0.05, y=-0.08, xanchor='left',
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=80, redraw=True),
                                      fromcurrent=True)]),
                dict(label='⏸ Pausa',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')]),
            ],
        )],
        sliders=[dict(
            active=0, steps=slider_steps,
            x=0.05, len=0.9, xanchor='left',
            y=-0.14, yanchor='top',
            currentvalue=dict(prefix='Tiempo: ', font=dict(size=13)),
            pad=dict(t=40),
        )],
    )
)
fig_39.show()

---
## 6. Animaciones con Matplotlib

Además de la visualización interactiva con Plotly, generamos animaciones con `matplotlib`
en dos modalidades:

### 6.1 Animación 2D (proyección XY)

Se anima la proyección sobre el plano XY de las 39 estrellas con un **rastro** de las
últimas $N_{\rm trail}$ posiciones. Esta visualización resalta la diferencia de velocidades
entre el disco negro (interior, rápido) y el disco rojo (exterior, lento).

> **Para visualizar in situ** se usa `HTML(anim.to_jshtml())`.
> **Para exportar a video** (10 s a 30 fps = 300 fotogramas) se usa `FFMpegWriter`.

In [ ]:
# ─── Parámetros de la animación 2D ───────────────────────────────────────────
N_FRAMES_2D  = 300          # fotogramas totales (10 s × 30 fps)
FPS_2D       = 30
TRAIL_LEN    = 25           # longitud del rastro (en pasos de simulación)

frame_idx_2d = np.linspace(0, N_pasos_39 - 1, N_FRAMES_2D, dtype=int)
trail_step   = max(1, int(N_pasos_39 / (N_FRAMES_2D * 3)))

# Colores
c39_list = [_col39(i) for i in range(n_cuerpos_39)]
sizes_39 = [18 if i == 0 else 6 for i in range(n_cuerpos_39)]
markers_39 = ['*' if i == 0 else 'o' for i in range(n_cuerpos_39)]

# Calcular límites del plot
all_x = rs_39[:, :, 0].ravel()
all_y = rs_39[:, :, 1].ravel()
lim   = max(np.percentile(np.abs(all_x), 99.5), np.percentile(np.abs(all_y), 99.5)) * 1.1

# ─── Figura de animación ─────────────────────────────────────────────────────
fig_anim2d, ax_anim2d = plt.subplots(figsize=(8, 8))
ax_anim2d.set_facecolor('#020210')
fig_anim2d.patch.set_facecolor('#020210')
ax_anim2d.set_xlim(-lim, lim); ax_anim2d.set_ylim(-lim, lim)
ax_anim2d.set_aspect('equal')
ax_anim2d.set_xlabel('$x$ (mpc)'); ax_anim2d.set_ylabel('$y$ (mpc)')
ax_anim2d.set_title('Cúmulo S — proyección XY', color='white')
ax_anim2d.grid(True, alpha=0.25)

# Objetos gráficos persistentes
trail_lines = [ax_anim2d.plot([], [], color=c39_list[i], lw=0.7, alpha=0.45)[0]
               for i in range(n_cuerpos_39)]
dot_artists = [ax_anim2d.plot([], [], color=c39_list[i],
               marker=markers_39[i], ms=sizes_39[i], ls='none',
               markeredgecolor='white', markeredgewidth=0.4)[0]
               for i in range(n_cuerpos_39)]
time_text = ax_anim2d.text(0.02, 0.96, '', transform=ax_anim2d.transAxes,
                            color='#FFFFFFAA', fontsize=10)

# Leyenda compacta
ax_anim2d.scatter([], [], c='#4FC3F7', label='Disco negro', s=20)
ax_anim2d.scatter([], [], c='#EF5350', label='Disco rojo',  s=20)
ax_anim2d.scatter([], [], c='gold',    label='Sgr A*',      s=100, marker='*')
ax_anim2d.legend(fontsize=8, loc='upper right')

def _init_2d():
    for tl, da in zip(trail_lines, dot_artists):
        tl.set_data([], [])
        da.set_data([], [])
    time_text.set_text('')
    return trail_lines + dot_artists + [time_text]

def _update_2d(frame):
    tidx   = frame_idx_2d[frame]
    t_start = max(0, tidx - TRAIL_LEN * trail_step)
    t_range = range(t_start, tidx + 1, max(1, trail_step))
    for i, (tl, da) in enumerate(zip(trail_lines, dot_artists)):
        tl.set_data(rs_39[i, list(t_range), 0], rs_39[i, list(t_range), 1])
        da.set_data([rs_39[i, tidx, 0]], [rs_39[i, tidx, 1]])
    time_text.set_text(f't = {ts_39[tidx]:.1f} yr')
    return trail_lines + dot_artists + [time_text]

anim_2d = FuncAnimation(fig_anim2d, _update_2d, frames=N_FRAMES_2D,
                         init_func=_init_2d, blit=True, interval=1000//FPS_2D)
plt.close(fig_anim2d)  # no mostrar figura estática

# Visualización in situ
print("Generando HTML de la animación 2D (puede tardar ~30 s)...")
html_2d = anim_2d.to_jshtml(fps=FPS_2D, default_mode='loop')
print("¡Listo! Mostrando animación in situ:")
HTML(html_2d)

### 6.2 Exportación de la animación 2D a video MP4

La celda siguiente guarda la animación en el archivo `s_stars_2d_animation.mp4`
(10 segundos, 30 fps, resolución 800×800 px a 150 dpi).

En Google Colab, la última línea descarga automáticamente el archivo al equipo local.
Si se ejecuta fuera de Colab, simplemente se omite la descarga.

In [ ]:
import os
# ─── Exportación de la animación 2D a MP4 ────────────────────────────────────
import subprocess, shutil

video_2d_path = 's_stars_2d_animation.mp4'

if shutil.which('ffmpeg'):
    writer_2d = FFMpegWriter(fps=FPS_2D,
                              metadata=dict(title='S-Stars 2D XY Animation',
                                            artist='ModelamientoNCuerpos'),
                              extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
                                          '-crf', '20'])
    print(f"Guardando animación 2D → {video_2d_path}  ({N_FRAMES_2D} frames @ {FPS_2D} fps = 10 s)")
    anim_2d.save(video_2d_path, writer=writer_2d, dpi=150)
    print(f"¡Guardado! Tamaño: {os.path.getsize(video_2d_path)/1024:.1f} KB")
else:
    # Fallback a GIF con Pillow cuando ffmpeg no está disponible
    video_2d_path = 's_stars_2d_animation.gif'
    writer_2d = PillowWriter(fps=FPS_2D)
    print(f"ffmpeg no encontrado. Guardando como GIF → {video_2d_path}")
    anim_2d.save(video_2d_path, writer=writer_2d, dpi=100)
    print("¡GIF guardado!")

# Descarga automática en Google Colab
try:
    from google.colab import files
    files.download(video_2d_path)
    print(f"Descarga iniciada: {video_2d_path}")
except ImportError:
    print(f"Archivo disponible localmente: {video_2d_path}")

### 6.3 Animación 3D con matplotlib

Se anima el sistema completo en el espacio tridimensional. Cada fotograma muestra
el marcador instantáneo de los 40 cuerpos con un rastro de sus posiciones recientes.
La cámara apunta desde un ángulo elevado para mostrar la estructura en 3D del cúmulo.

In [ ]:
# ─── Parámetros de la animación 3D ───────────────────────────────────────────
N_FRAMES_3D = 300          # 10 s × 30 fps
FPS_3D      = 30
TRAIL_3D    = 20           # longitud del rastro en pasos

frame_idx_3d = np.linspace(0, N_pasos_39 - 1, N_FRAMES_3D, dtype=int)
trail_step3  = max(1, int(N_pasos_39 / (N_FRAMES_3D * 3)))

all_xyz = rs_39.reshape(-1, 3)
lim3    = np.percentile(np.abs(all_xyz), 99.5) * 1.1

# ─── Figura 3D ────────────────────────────────────────────────────────────────
fig_anim3d = plt.figure(figsize=(9, 8))
ax_anim3d  = fig_anim3d.add_subplot(111, projection='3d')
ax_anim3d.set_facecolor('#020210')
fig_anim3d.patch.set_facecolor('#020210')
ax_anim3d.set_xlim(-lim3, lim3); ax_anim3d.set_ylim(-lim3, lim3); ax_anim3d.set_zlim(-lim3, lim3)
ax_anim3d.set_xlabel('$x$ (mpc)'); ax_anim3d.set_ylabel('$y$ (mpc)'); ax_anim3d.set_zlabel('$z$ (mpc)')
ax_anim3d.set_title('Cúmulo S — animación 3D', color='white')
ax_anim3d.tick_params(colors='#aaaacc')

trails3d = [ax_anim3d.plot([], [], [], color=c39_list[i], lw=0.8, alpha=0.5)[0]
            for i in range(n_cuerpos_39)]
dots3d   = [ax_anim3d.plot([], [], [], color=c39_list[i],
                            marker=markers_39[i], ms=sizes_39[i],
                            ls='none', markeredgecolor='white', markeredgewidth=0.35)[0]
            for i in range(n_cuerpos_39)]
ttext3d  = ax_anim3d.text2D(0.02, 0.97, '', transform=ax_anim3d.transAxes,
                              color='#FFFFFFAA', fontsize=10)

def _init_3d():
    for tl, da in zip(trails3d, dots3d):
        tl.set_data([], []); tl.set_3d_properties([])
        da.set_data([], []); da.set_3d_properties([])
    ttext3d.set_text('')
    return trails3d + dots3d + [ttext3d]

def _update_3d(frame):
    tidx   = frame_idx_3d[frame]
    t_start = max(0, tidx - TRAIL_3D * trail_step3)
    t_range = list(range(t_start, tidx + 1, max(1, trail_step3)))
    for i, (tl, da) in enumerate(zip(trails3d, dots3d)):
        tl.set_data(rs_39[i, t_range, 0], rs_39[i, t_range, 1])
        tl.set_3d_properties(rs_39[i, t_range, 2])
        da.set_data([rs_39[i, tidx, 0]], [rs_39[i, tidx, 1]])
        da.set_3d_properties([rs_39[i, tidx, 2]])
    ttext3d.set_text(f't = {ts_39[tidx]:.1f} yr')
    return trails3d + dots3d + [ttext3d]

anim_3d = FuncAnimation(fig_anim3d, _update_3d, frames=N_FRAMES_3D,
                         init_func=_init_3d, blit=False, interval=1000//FPS_3D)
plt.close(fig_anim3d)

print("Generando HTML de la animación 3D (puede tardar ~45 s)...")
html_3d = anim_3d.to_jshtml(fps=FPS_3D, default_mode='loop')
print("¡Listo! Mostrando animación 3D in situ:")
HTML(html_3d)

### 6.4 Exportación de la animación 3D a video MP4

In [ ]:
import os
# ─── Exportación de la animación 3D a MP4 ────────────────────────────────────
video_3d_path = 's_stars_3d_animation.mp4'

if shutil.which('ffmpeg'):
    writer_3d = FFMpegWriter(fps=FPS_3D,
                              metadata=dict(title='S-Stars 3D Animation'),
                              extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
                                          '-crf', '22'])
    print(f"Guardando animación 3D → {video_3d_path}  ({N_FRAMES_3D} frames @ {FPS_3D} fps = 10 s)")
    anim_3d.save(video_3d_path, writer=writer_3d, dpi=120)
    print(f"¡Guardado! Tamaño: {os.path.getsize(video_3d_path)/1024:.1f} KB")
else:
    video_3d_path = 's_stars_3d_animation.gif'
    writer_3d = PillowWriter(fps=FPS_3D)
    print(f"ffmpeg no encontrado. Guardando como GIF → {video_3d_path}")
    anim_3d.save(video_3d_path, writer=writer_3d, dpi=80)
    print("¡GIF guardado!")

try:
    from google.colab import files
    files.download(video_3d_path)
    print(f"Descarga iniciada: {video_3d_path}")
except ImportError:
    print(f"Archivo disponible localmente: {video_3d_path}")

---
## 7. Análisis Cuantitativo de Perturbaciones — 39 Estrellas

Se repite la comparación N-cuerpos vs. 2-cuerpos para el conjunto completo.
El cociente de fuerzas teórico es:

$$\frac{F_\text{estrellas}}{F_{\text{Sgr A*}}} \approx
\frac{N_{\rm est}\,\langle m \rangle}{4\,M_{\text{Sgr A*}}} \sim \mathcal{O}(10^{-5})$$

El gráfico de barras muestra la desviación relativa máxima $\max_t (\Delta r/|r|)$
para cada estrella, separando el disco negro del rojo.

> **Nota de rendimiento:** el bucle de 39 integraciones kepleranas puede tomar $\sim 2$ minutos
> en Colab (CPU estándar).

In [ ]:
# ─── 2-cuerpos para las 39 estrellas ─────────────────────────────────────────
# rs_2b_i[1, :, :] = trayectoria kepleriana de la estrella i
print(f"Calculando {N_estrellas} trayectorias kepleranas (puede tardar ~2 min en Colab)...")
max_dr_rel = []

for i, (nombre, estado, masa) in enumerate(zip(nombres_todos, estados_todos, masas_todos)):
    sistema_2b = [
        dict(m=mu_SgrA,        r=[0., 0., 0.],     v=[0., 0., 0.]),
        dict(m=G_canon * masa,  r=list(estado[:3]), v=list(estado[3:])),
    ]
    rs_2b_i, _, _, _, _ = pc.ncuerpos_solucion(sistema_2b, ts_39)
    r_nb_i  = rs_39[i + 1, :, :]          # trayectoria N+1 cuerpos
    r_2b_i  = rs_2b_i[1, :, :]            # trayectoria kepleriana
    dr_rel  = (np.linalg.norm(r_nb_i - r_2b_i, axis=1) /
               (np.linalg.norm(r_nb_i, axis=1) + 1e-30)).max()
    max_dr_rel.append(dr_rel)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N_estrellas} completadas...")

print("¡Completado!")

# ─── Gráfico de barras ────────────────────────────────────────────────────────
bar_colors = ['steelblue' if d == 'black' else 'tomato' for d in disco_todos]

fig_pert, ax_pert = plt.subplots(figsize=(14, 5))
ax_pert.bar(nombres_todos, max_dr_rel, color=bar_colors)
ax_pert.set_yscale('log')
ax_pert.set(xlabel="Estrella",
            ylabel=r"Desviación relativa máx.  $\Delta r / |r|$",
            title="Perturbaciones N-cuerpos vs. Kepler puro (2-cuerpos)")
ax_pert.tick_params(axis='x', rotation=90)
ax_pert.grid(True, axis='y', alpha=0.5)

ratio_total = np.sum(masas_todos) / 4.3e6
ax_pert.axhline(ratio_total, ls='--', color='gray')
ax_pert.legend(handles=[
    mpatches.Patch(color='steelblue', label='Disco negro (inner cluster)'),
    mpatches.Patch(color='tomato',    label='Disco rojo  (outer cluster)'),
    plt.Line2D([0], [0], ls='--', color='gray',
               label=fr'$M_{{\rm est}}/M_{{\rm Sgr\,A*}} \approx {ratio_total:.1e}$'),
])
plt.tight_layout(); plt.show()

# ─── Resumen numérico ─────────────────────────────────────────────────────────
i_max = int(np.argmax(max_dr_rel)); i_min = int(np.argmin(max_dr_rel))
print(f"\nResumen de perturbaciones estrella–estrella:")
print(f"  M_total_estrellas / M_SgrA  = {ratio_total:.2e}")
print(f"  Desviación relativa media   = {np.mean(max_dr_rel):.2e}")
print(f"  Desviación máxima           = {max_dr_rel[i_max]:.2e}  ({nombres_todos[i_max]})")
print(f"  Desviación mínima           = {max_dr_rel[i_min]:.2e}  ({nombres_todos[i_min]})")
print(f"\n→ Perturbaciones de orden {np.mean(max_dr_rel):.0e}, "
      f"coherente con la relación de masas M_est/M_SgrA ~ {ratio_total:.0e}.")

### Visualización detallada de perturbaciones

In [ ]:
# ─── Figura 8: Perturbaciones ordenadas + comparativa por disco ──────────────
sorted_idx  = np.argsort(max_dr_rel)[::-1]  # orden descendente
sorted_names = [nombres_todos[i] for i in sorted_idx]
sorted_dr    = [max_dr_rel[i]    for i in sorted_idx]
sorted_disks = [disco_todos[i]   for i in sorted_idx]
sorted_cols  = ['#4FC3F7' if d=='black' else '#EF5350' for d in sorted_disks]

fig_pert2, axes_pert = plt.subplots(1, 2, figsize=(15, 6))
fig_pert2.suptitle("Perturbaciones N-cuerpos vs. Kepler puro — 39 estrellas S", fontsize=13)

ax_bar = axes_pert[0]
bars = ax_bar.barh(sorted_names, sorted_dr, color=sorted_cols,
                   edgecolor='white', lw=0.4)
ax_bar.set_xscale('log')
ax_bar.set(xlabel=r'$\max_t(\Delta r / |r|)$  (adim.)',
           title='Desviación relativa máxima (orden descendente)')
ax_bar.invert_yaxis()
ax_bar.grid(True, axis='x', alpha=0.4)
patch_b = mpatches.Patch(color='#4FC3F7', label='Disco negro')
patch_r = mpatches.Patch(color='#EF5350', label='Disco rojo')
ax_bar.legend(handles=[patch_b, patch_r], fontsize=9)

# ─── Distribución logarítmica por disco ──────────────────────────────────────
ax_hist = axes_pert[1]
dr_black = [max_dr_rel[i] for i,d in enumerate(disco_todos) if d=='black']
dr_red   = [max_dr_rel[i] for i,d in enumerate(disco_todos) if d=='red']
bins_log = np.logspace(np.log10(min(max_dr_rel)*0.5), np.log10(max(max_dr_rel)*2.0), 15)
ax_hist.hist(dr_black, bins=bins_log, alpha=0.75, color='#4FC3F7', label='Disco negro', edgecolor='white', lw=0.4)
ax_hist.hist(dr_red,   bins=bins_log, alpha=0.75, color='#EF5350', label='Disco rojo',  edgecolor='white', lw=0.4)
ax_hist.set_xscale('log')
ax_hist.set(xlabel=r'$\max_t(\Delta r / |r|)$',
            ylabel='Número de estrellas',
            title='Histograma de perturbaciones por disco')
ax_hist.legend(fontsize=9); ax_hist.grid(True, axis='y', alpha=0.4)

# Estadísticas
print(f"Disco negro  — mediana: {np.median(dr_black):.2e}  máx: {max(dr_black):.2e}")
print(f"Disco rojo   — mediana: {np.median(dr_red):.2e}  máx: {max(dr_red):.2e}")
print(f"Global       — mediana: {np.median(max_dr_rel):.2e}  máx: {max(max_dr_rel):.2e}  mín: {min(max_dr_rel):.2e}")

plt.tight_layout()
plt.savefig('perturbation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig. 8 — Análisis de perturbaciones guardada.")

---
## 8. Conclusiones

### 8.1 Dinámica Newtoniana base

Las **39 estrellas del cúmulo S** mantienen sus órbitas elípticas kepleranas durante los 100 años
simulados con el integrador de Adams-Moulton de `pymcel`.

- Las estrellas del **disco negro** (cúmulo interno, $a < 50$ mpc) completan múltiples
  órbitas: **S2** ($a = 5.03$ mpc, $e = 0.887$, $T \approx 16$ yr) realiza $\sim$6 órbitas
  completas, alcanzando velocidades de periastro $\gtrsim 7500$ km/s.
- Las estrellas del **disco rojo** (cúmulo externo) con $T \gtrsim 10^3$ yr apenas avanzan
  una fracción de su período orbital en la ventana de 100 años.
- El **error relativo de energía** máximo es $< 10^{-5}$ en todos los casos, confirmando
  la precisión del integrador.

### 8.2 Tercera Ley de Kepler

La relación $T^2 \propto a^3$ se verifica numéricamente con excelente acuerdo respecto al valor
teórico. La pendiente $3/2$ en el diagrama $\log T$ vs $\log a$ se reproduce en todo el rango de
semiejes (desde $\sim 5$ mpc hasta $\sim 400$ mpc), confirmando la dominancia de Sgr A*.

### 8.3 Perturbaciones de N-cuerpos

La masa total de las estrellas es $\sum m_i \approx 430\,M_\odot$, frente a
$M_{\rm Sgr\,A*} \approx 4.3 \times 10^6\,M_\odot$:

$$\frac{F_{\rm estrellas}}{F_{\rm Sgr\,A*}} \sim \frac{M_{\rm est}}{M_{\rm Sgr\,A*}} \sim 10^{-4}$$

Numéricamente, la desviación relativa máxima acumulada tras 100 años oscila entre $10^{-6}$ y
$10^{-2}$ según la estrella.

- Las **más perturbadas** son estrellas del disco rojo con períodos largos: permanecen más tiempo
  en las regiones externas donde las distancias inter-estelares son comparables a las distancias
  a Sgr A*, amplificando el efecto de las perturbaciones mutuas.
- Las **menos perturbadas** son las estrellas internas de período corto (S2, S55): pasan poco
  tiempo en las regiones externas y la fuerza de Sgr A* domina en toda su órbita.

### 8.4 Conservación del momento angular

Los vectores de momento angular específico $\mathbf{h}_i$ presentan variaciones relativas
$\Delta |h|/h_0 < 10^{-4}$ en 100 años, consistentes con la magnitud de las perturbaciones.
La dirección del vector orbital (polo orbital) experimenta una precesión muy lenta,
detectable numéricamente pero físicamente insignificante en escalas de décadas.

### 8.5 Perspectivas y trabajos futuros

1. **Precesión relativista:** incluir la corrección post-newtoniana de Schwarzschild
   (detectada observacionalmente en S2 en 2020 por la colaboración GRAVITY).
2. **Escalas de tiempo largas:** extender la integración a $10^4$–$10^5$ años para
   observar la difusión orbital por perturbaciones acumuladas.
3. **Masa extendida:** agregar una distribución de materia oscura o estrellas de neutrones
   alrededor de Sgr A* y estudiar su efecto sobre las órbitas de las estrellas S.
4. **Monte Carlo de masas:** variar las masas estelares dentro de sus incertidumbres
   observacionales y estudiar la sensibilidad de las perturbaciones.
5. **Integrador simpléctico:** comparar con el integrador de Leapfrog/Yoshida para
   evaluar la conservación de energía a largo plazo.